In [ ]:
# ============================================================
# "THE PRICE IS RIGHT" — Used Car Edition
# Fine-tuning GPT-4.1-nano on used car listings
# ============================================================
#
# DAY 5: Fine-tuning a Frontier Model
#
# We will fine-tune a private variant of GPT-4.1-nano
# to predict used car prices from a short text description.
# ============================================================

# ── 0. Install dependencies ───────────────────────────────────────────────────
!pip install  datasets pandas scikit-learn openai python-dotenv tqdm



In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import os
import re
import io
import json
import time
from datetime import datetime

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from openai import OpenAI
from tqdm import tqdm



In [ ]:
# ── 2. Load & preprocess the dataset ─────────────────────────────────────────
dataset    = load_dataset("imbalet/used_cars")
train_full = dataset["train"]
print(train_full[0])

SYSTEM_PROMPT = (
    "You are a used car price estimator for the Russian market. "
    "Given a short description of a car, respond with only the estimated price "
    "in rubles as a plain number, no currency symbol, no explanation."
)


def make_summary(ex):
    # Use displacement (numeric) for engine size, fall back to engine fuel type
    engine = (
        f"{ex['displacement'] / 1000:.1f}L"   # displacement is in cc, convert to litres
        if ex.get("displacement")
        else ex.get("engine", "")
    )
    text = f"{ex['year']} {ex['mark']} {ex['model']} {engine} {ex['mileage']} miles"
    if ex.get("transmission"):
        text += f" {ex['transmission'].lower()}"
    if ex.get("gear_type"):
        text += f" {ex['gear_type'].lower().replace('_', ' ')}"
    if ex.get("power"):
        text += f" {ex['power']}hp"
    return {"summary": text.strip(), "price": ex["price"]}


dataset = train_full.map(make_summary)
dataset = dataset.filter(lambda x: x["price"] > 100 and x["summary"] != "")



In [ ]:
# ── 3. Train / val / test split ───────────────────────────────────────────────
df = dataset.to_pandas()
train, temp = train_test_split(df, test_size=0.2, random_state=42)
val, test   = train_test_split(temp, test_size=0.5, random_state=42)
print(f"Loaded {len(train):,} training rows, {len(val):,} validation rows, {len(test):,} test rows")



In [ ]:
# ── 4. Data size ──────────────────────────────────────────────────────────────
#
# OpenAI recommends fine-tuning with 50-100 examples.
# We cap at 500 train / 50 val — more than enough for GPT-4.1-nano
# to learn car pricing patterns at minimal cost.
#
# Cost reference: ~500 training examples × 1 epoch ≈ < $0.10

FINE_TUNE_TRAIN_SIZE = 2000   # was 500 — you have 345k rows, use them
FINE_TUNE_VAL_SIZE   = 200    # was 50

fine_tune_train = train.head(FINE_TUNE_TRAIN_SIZE)
fine_tune_val   = val.head(FINE_TUNE_VAL_SIZE)

print(f"Fine-tune train size : {len(fine_tune_train)}")
print(f"Fine-tune val size   : {len(fine_tune_val)}")



In [ ]:
# ── 5. Build chat messages ────────────────────────────────────────────────────

def messages_for(row):
    """Return the full chat messages list for a training row (includes assistant turn)."""
    message = (
        f"Estimate the price of this used car in rubles. "
        f"Respond with the number only, no symbols.\n\n{row['summary']}"
    )
    return [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": message},
        {"role": "assistant", "content": str(int(row["price"]))},  # plain ruble number
    ]


def test_messages_for(row):
    """Return chat messages WITHOUT the assistant turn (for inference)."""
    message = (
        f"Estimate the price of this used car in rubles. "
        f"Respond with the number only, no symbols.\n\n{row['summary']}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": message},
    ]


# Quick sanity check
print(messages_for(fine_tune_train.iloc[0]))



In [ ]:
# ── 6. Convert to JSONL and write to disk ────────────────────────────────────

os.makedirs("jsonl", exist_ok=True)


def make_jsonl(df):
    lines = []
    for _, row in df.iterrows():
        lines.append(json.dumps({"messages": messages_for(row)}))
    return "\n".join(lines)


def write_jsonl(df, filename):
    with open(filename, "w") as f:
        f.write(make_jsonl(df))
    print(f"Wrote {len(df)} rows → {filename}")


write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")
write_jsonl(fine_tune_val,   "jsonl/fine_tune_val.jsonl")

# Preview first record
with open("jsonl/fine_tune_train.jsonl") as f:
    print(f.readline())



In [ ]:
# ── 7. Upload files to OpenAI ─────────────────────────────────────────────────

openai_api_key = os.getenv('OPENAI_API_KEY')

def upload_with_progress(path, purpose="fine-tune"):
    """Upload a file to OpenAI with a live byte-level progress bar."""
    file_size = os.path.getsize(path)
    filename  = os.path.basename(path)

    buffer = io.BytesIO()
    with open(path, "rb") as f:
        with tqdm(total=file_size, desc=f"Uploading {filename}",
                  unit="B", unit_scale=True, unit_divisor=1024) as pbar:
            while chunk := f.read(8192):
                buffer.write(chunk)
                pbar.update(len(chunk))

    buffer.seek(0)
    buffer.name = filename

    result = openai.files.create(file=buffer, purpose=purpose)
    print(f"✓ Uploaded → file ID: {result.id}")
    return result


train_file = upload_with_progress("jsonl/fine_tune_train.jsonl")
val_file   = upload_with_progress("jsonl/fine_tune_val.jsonl")

# You can also browse your uploaded files at:
# https://platform.openai.com/storage/files/



In [ ]:
# ── 8. Create fine-tuning job ─────────────────────────────────────────────────
#
# batch_size=8  → ~63 steps instead of 500 (much faster than batch_size=1)
# n_epochs=1    → single pass, avoids overfitting on small dataset

job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 3, "batch_size": 8},  # 3 epochs for better convergence
    suffix="car_pricer",
)
print("Fine-tuning job ID:", job.id)

# You can also monitor the job at:
# https://platform.openai.com/finetune



In [ ]:
# ── 9. Monitor fine-tuning job ────────────────────────────────────────────────

def monitor_fine_tuning(job_id, poll_interval=5):
    seen_events = set()
    pbar = tqdm(total=100, desc="Fine-tuning progress", unit="step")

    while True:
        job_info = openai.fine_tuning.jobs.retrieve(job_id)
        status   = job_info.status

        events = openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)
        for e in reversed(events.data):
            if e.id not in seen_events:
                ts  = datetime.fromtimestamp(e.created_at)
                msg = e.message
                tqdm.write(f"{ts} | {e.level.upper()} | {msg}")

                step_match = re.search(r"Step (\d+)/(\d+)", msg)
                if step_match:
                    current_step = int(step_match.group(1))
                    total_steps  = int(step_match.group(2))
                    pbar.total   = total_steps
                    pbar.n       = current_step
                    pbar.refresh()

                seen_events.add(e.id)

        if status == "succeeded":
            pbar.n = pbar.total or 100
            pbar.refresh()
            tqdm.write("\n✓ Fine-tuning completed successfully!")
            break
        elif status in ("failed", "cancelled"):
            tqdm.write(f"\n✗ Fine-tuning ended with status: {status}")
            break

        time.sleep(poll_interval)

    pbar.close()
    return openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model


fine_tuned_model_name = monitor_fine_tuning(job.id)
print("Fine-tuned model:", fine_tuned_model_name)

# ── 10. Test the fine-tuned model ─────────────────────────────────────────────

def predict_price(row):
    """Run inference with the fine-tuned model and return the raw response string."""
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(row),
        max_tokens=10,
    )
    return response.choices[0].message.content


def parse_price(raw: str) -> float:
    """Strip currency symbols/commas and parse to float."""
    cleaned = re.sub(r"[^\d.]", "", raw)
    try:
        return float(cleaned)
    except ValueError:
        return 0.0


# Quick sanity check on first test example
sample = test.iloc[0]
raw    = predict_price(sample)
print(f"Actual    : {int(sample['price']):,} RUB")
print(f"Predicted : {raw}  →  parsed: {parse_price(raw):,.0f} RUB")

# ── 11. Evaluate on the full test set ─────────────────────────────────────────
#
# We measure accuracy as: predictions within ±20% of the actual price
# (same methodology as the Amazon capstone)

def evaluate(df, n=100, workers=10):
    """Evaluate accuracy in parallel — predictions within ±20% count as correct."""
    from concurrent.futures import ThreadPoolExecutor, as_completed

    subset  = df.head(n)
    results = {}

    def run(idx_row):
        idx, row = idx_row
        raw       = predict_price(row)
        predicted = parse_price(raw)
        actual    = row["price"]
        hit       = actual > 0 and abs(predicted - actual) / actual <= 0.20
        return idx, hit

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(run, (idx, row)): idx
                   for idx, row in subset.iterrows()}
        with tqdm(total=n, desc="Evaluating") as pbar:
            for future in as_completed(futures):
                idx, hit = future.result()
                results[idx] = hit
                pbar.update(1)

    correct  = sum(results.values())
    accuracy = correct / n * 100
    print(f"\nAccuracy (±20% threshold): {accuracy:.2f}%  ({correct}/{n} correct)")
    return accuracy


accuracy = evaluate(test, n=100)

# ── Benchmark reference ───────────────────────────────────────────────────────
# Accuracy scores logged here as you experiment:
#
# v1 — bad summary (engine type not size), dollar prices, 500 examples : 24%
# v2 — rubles + year added, 500 examples, 1 epoch                      : 29%
# v3 — fix displacement cc→L, 2000 examples, 3 epochs                  : TBD

In [ ]:
# ── 10. Test the fine-tuned model ─────────────────────────────────────────────

def predict_price(row):
    """Run inference with the fine-tuned model and return the raw response string."""
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(row),
        max_tokens=10,
    )
    return response.choices[0].message.content


def parse_price(raw: str) -> float:
    """Strip currency symbols/commas and parse to float."""
    cleaned = re.sub(r"[^\d.]", "", raw)
    try:
        return float(cleaned)
    except ValueError:
        return 0.0


# Quick sanity check on first test example
sample = test.iloc[0]
raw    = predict_price(sample)
print(f"Actual    : {int(sample['price']):,} RUB")
print(f"Predicted : {raw}  →  parsed: {parse_price(raw):,.0f} RUB")



In [ ]:
# ── 11. Evaluate on the full test set ─────────────────────────────────────────
#
# We measure accuracy as: predictions within ±20% of the actual price
# (same methodology as the Amazon capstone)

def evaluate(df, n=100, workers=10):
    """Evaluate accuracy in parallel — predictions within ±20% count as correct."""
    from concurrent.futures import ThreadPoolExecutor, as_completed

    subset  = df.head(n)
    results = {}

    def run(idx_row):
        idx, row = idx_row
        raw       = predict_price(row)
        predicted = parse_price(raw)
        actual    = row["price"]
        hit       = actual > 0 and abs(predicted - actual) / actual <= 0.20
        return idx, hit

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(run, (idx, row)): idx
                   for idx, row in subset.iterrows()}
        with tqdm(total=n, desc="Evaluating") as pbar:
            for future in as_completed(futures):
                idx, hit = future.result()
                results[idx] = hit
                pbar.update(1)

    correct  = sum(results.values())
    accuracy = correct / n * 100
    print(f"\nAccuracy (±20% threshold): {accuracy:.2f}%  ({correct}/{n} correct)")
    return accuracy


accuracy = evaluate(test, n=100)

# ── Benchmark reference ───────────────────────────────────────────────────────
# Accuracy scores logged here as you experiment:
#
# v1 — bad summary (engine type not size), dollar prices, 500 examples : 24%
# v2 — rubles + year added, 500 examples, 1 epoch                      : 29%
# v3 — fix displacement cc→L, 2000 examples, 3 epochs                  : 63%